In [1]:
import numpy as np
import sys
import tabix
import json
import h5py
import scipy.stats as stats
import time

In [2]:
def roiURL(roi, top_regions, detail_level, target_factors_str,
           link_type='dual'):
    today = date.today()
    roiurl_name = ('region_narration_simsearch_regions_roi_' + str(roi) +
                   '_detail_level_' + str(detail_level) + '_factors_' +
                   target_factors_str + '_scale_25kb_CT' +
                   today.strftime("%Y%m%d") + '.bed')
    bed_path_name = os.path.join('/home/ctrader/public_html/', roiurl_name)
    np.savetxt(bed_path_name, top_regions, delimiter='\t', fmt='%s')
    link_path1 = ('https://epilogos.altius.org:1234/?application=viewer&' +
                  'sampleSet=vC&mode=single&genome=hg38&model=18&complex' +
                  'ity=KL&group=all&chrLeft=chr1&chrRight=chr1&start=800' +
                  '0&stop=35000&roiURL=https%3A%2F%2Fencode%3Acollabor8%' +
                  '40resources.altius.org%2F~ctrader%2F' + roiurl_name)
    link_path2 = ('https://index.altius.org:1234/?application=viewer&mod' +
                  'e=pd&chrLeft=chr11&chrRight=chr11&start=47329800&stop' +
                  '=47356800&roiURL=https%3A%2F%2Fencode%3Acollabor8%40r' +
                  'esources.altius.org%2F~ctrader%2F' + roiurl_name)
    link_path_dual = ('https://index.altius.org:3000/?application=viewer' +
                      '&mode=cf&&roiURL=https%3A%2F%2Fencode%3Acollabor8' +
                      '%40resources.altius.org%2F~ctrader%2F' + roiurl_name)
    if link_type == 'dual':
        return link_path_dual
    else:
        return link_path1, link_path2
    
def load_per(per_f):
    hdf5_f = h5py.File(per_f, 'r')
    percentile = hdf5_f['percentile'][:]
    factors = np.char.decode(hdf5_f['factors'][:])
    color = np.char.decode(hdf5_f['colors'][:])
    hdf5_f.close()
    return percentile, factors, color


def top_region_sig(percents, top_regions):
    # mean and std of factor percentiles in all regions
    all_mean = np.mean(percents, axis=0)
    all_std = np.std(percents, axis=0)

    # mean of factor percentiles in most similar regions
    top_region_mean = np.mean(percents[top_regions[:, 1:]], axis=1)

    # Z score
    Z = (top_region_mean - all_mean) / (all_std / np.sqrt(top_regions.shape[1] - 1))

    # p-value (one sided)
    pval = stats.norm.sf(Z)

    return pval, Z

def simsearch(percents, roi, included_factors):

    # factors the query region rank highest in
    query_factor_order = np.argsort(-percents[roi])

    # factors to search over based on detail level
    target_factors = query_factor_order[
        np.isin(query_factor_order, included_factors)
    ]

    # convolution
    roi_percents = percents[roi, target_factors]
    roi_conv_all = percents[:, target_factors] * roi_percents

    # fusion
    roi_sim_all = []
    for dl in range(1, 35):
        roi_sim = 1 - np.prod(1 - roi_conv_all[:, :dl], axis=1)
        roi_sim_all.append(roi_sim)
    roi_sim_all = np.array(roi_sim_all)

    # most similar regions
    top_regions = []
    top_regions_pre = np.argsort(-roi_sim_all, axis=1)[:, :101]
    for i in range(34):
        top_regions.append(
            np.append(
                roi,
                top_regions_pre[i][top_regions_pre[i] != roi][:100]
            )
        )
    top_regions = np.array(top_regions)

    # factor percentile significance
    pval, Z = top_region_sig(percents, top_regions)
    
    rows = (
        np.ones(top_regions.shape, dtype=int) *
        np.arange(top_regions.shape[0]).reshape(-1, 1)
    )

    return (top_regions, roi_sim_all[rows, top_regions],
            percents[top_regions], query_factor_order, target_factors, pval, Z)

def narrate_detail_level(top_regions, roi_per, factor_order, metric_order, included_factors, detail_level, conversion_dict, metriccolors, thresh=0.9):
    query = top_regions[0]
    hits = top_regions[1:]
    factor_scores = roi_per[0] * roi_per  # > sqrt of thresh for query region

    sentences = []
    for region, factor_score in zip(top_regions, factor_scores):
        factor_score_sort = np.argsort(-factor_score)
        key_factors = factor_score_sort[
            ((factor_score[factor_score_sort] > thresh) &
             (np.isin(factor_score_sort, included_factors)))
        ][:detail_level]

        if len(key_factors) > 0:
            if region == query:
                sentence = '<div>This region is characterized by '
            else:
                sentence = '<div>This region is highly similar to the query region in terms of '
            key_factor_names = factor_order[key_factors]
            key_factor_names = key_factor_names.astype(object)
            for i in range(len(key_factor_names)):
                 key_factor_names[i] = conversion_dict[key_factor_names[i]]
            key_metric_names = metric_order[key_factors]

            string_factors = []
            for factor, metric in zip(key_factor_names, key_metric_names):
                if metric == 'chromatin states':
                    string_factor = metriccolors['chromatin states']
                elif metric == 'DHSs':
                    string_factor = metriccolors['DHSs']
                string_factor += factor + ' ' + metric + metriccolors['STANDARD']
                string_factors.append(string_factor)

            if len(string_factors) > 2:
                sentence += ', '.join(string_factors[:-1])
                sentence += ', and ' + string_factors[-1]
            elif len(string_factors) == 2:
                sentence += ' and '.join(string_factors)
            else:
                sentence += string_factors[0]
            sentence += '.</div>'
        else:
            sentence = '<div></div>'
        sentences.append(sentence)
    return sentences

In [3]:
datadir='/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
gz_file_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.rownum4.bed.gz'
regions_path = datadir + 'hg38.25kb.maxMean.CT20230118.sorted.bed'

DHS_per_f = datadir + 'regions_25kb_CT20230214.DHS.factornormNMF.sum_enrich_corank_percentile.hdf5'
epi_per_f = datadir + 'regions_25kb_CT20230214.epilogos.statesum_rank_percentile.hdf5'

genomic_location = 59157
included_factors_str = '[' + ','.join((np.arange(34)[np.arange(34)!=30].astype(str))) + ']'

In [14]:
t = time.time()

json_return_all = []
for genomic_location in range(1000):
    if genomic_location % 100 == 0:
        print(genomic_location)

    if str.isalpha(str(genomic_location)[0]):
        # load roi
        gz_tb = tabix.open(gz_file_path)
        roi = np.array(
            list(gz_tb.querys(genomic_location))
        )[:, 3].astype(int)[0]
    else:
        roi = int(genomic_location)

    # print(time.time()-t)

    # load region percentiles
    DHS_per, DHS_factors, DHS_color = load_per(DHS_per_f)
    epi_per, epi_factors, epi_color = load_per(epi_per_f)

    # print()
    # print(time.time()-t)

    # join data from each metric
    percents = np.hstack((DHS_per, epi_per))
    factor_order = np.append(DHS_factors, epi_factors)
    metric_order = np.append(np.array(['DHSs'] * len(DHS_factors)),
                             np.array(['chromatin states'] *
                                      len(epi_factors)))
    colors = np.append(DHS_color, epi_color)
    # conversion dictionary
    conversion_dict = {
        'EnhA1': 'active enhancer 1',
        'EnhA2': 'active enhancer 2',
        'EnhBiv': 'bivalent enhancer',
        'EnhG1': 'genic enhancer 1',
        'EnhG2': 'genic enhancer 2',
        'EnhWk': 'weak enhancer',
        'Het': 'heterochromatin',
        'Quies': 'quiescent',
        'ReprPC': 'repressed polycomb',
        'ReprPCWk': 'weak repressed polycomb',
        'TssA': 'active TSS',
        'TssBiv': 'bivalent TSS',
        'TssFlnk': 'flanking TSS',
        'TssFlnkD': 'downstream flanking TSS',
        'TssFlnkU': 'upstream flanking TSS',
        'Tx': 'strong transcription',
        'TxWk': 'weak transcription',
        'ZNF_Rpts': 'ZNF genes + repeats',
        'Cancer_epithelial': 'cancer/epithelial',
        'Cardiac': 'cardiac',
        'Digestive': 'digestive',
        'Lymphoid': 'lymphoid',
        'Musculoskeletal': 'musculoskeletal',
        'Myeloid_erythroid': 'myeloid/erythroid',
        'Neural': 'neural',
        'Organdevel_renal': 'organ development/renal',
        'Placental_trophoblast': 'placental/trophoblast',
        'Primitive_embryonic': 'primitive/embryonic',
        'Pulmonarydevel': 'pulmonary development',
        'Renal_cancer': 'renal/cancer',
        'StromalA': 'stromal A',
        'StromalB': 'stromal B',
        'Tissueinvariant': 'tissue invariant',
        'Vascular_endothelial': 'vascular/endothelial'
    }

    metriccolors = {
        'DHSs': '<span style="color:#0000CD">',
        'chromatin states': '<span style="color:#FF0000">',
        'STANDARD': '</span>',
        'BOLD': '<b>',
        'UNBOLD': '</b>'
    }

    # print(time.time()-t)

    # factors the user cares about
    included_factors = np.int_(
        included_factors_str.strip('[').strip(']').split(',')
    )

    # print(time.time()-t)

    # genomic coordinates for each region
    f = open(regions_path, 'r')
    regions = []
    for line in f.readlines():
        line = line.strip().split('\t')
        regions.append(line[0] + ':' + line[1] + '-' + line[2])
    f.close()

    # print(time.time()-t)

    # find most similar region inds
    (top_regions, roi_sim,
     roi_per, query_factor_order,
     target_factors, pval,
     Z) = simsearch(percents, roi, included_factors)

    # print()
    # print(time.time()-t)

    # dual track roiURL
    # roiurl = roiURL(roi, top_regions, detail_level,
    #                 '_'.join(target_factors.astype(str)))

    top_region_coords = np.array(regions)[top_regions]

    # print(time.time()-t)

    # narrate regions
    sentences = []
    for i in range(34):
        detail_level = i + 1
        sentences_pre = narrate_detail_level(top_regions[i], roi_per[i],
                                             factor_order, metric_order,
                                             included_factors, detail_level, conversion_dict,
                                             metriccolors)
        sentences.append(sentences_pre)
    sentences = np.array(sentences)

    # print(time.time()-t)

    # reorder based on query region
    roi_per = roi_per[:, :, query_factor_order]
    return_factor_order = factor_order[query_factor_order]
    return_metric_order = metric_order[query_factor_order]
    return_colors = colors[query_factor_order]
    pval = pval[:, query_factor_order]
    Z = Z[:, query_factor_order]

    # print(time.time()-t)

    # return_factor_order = return_factor_order.astype(object)
    for i in range(len(return_factor_order)):
        return_factor_order[i] = return_factor_order[i].replace('_', '/')
    #     return_factor_order[i] = metriccolors[return_metric_order[i]] + return_factor_order[i] + metriccolors['STANDARD']

    return_arr_full = []
    for dl in range(34):
        # for the roi, return results as json array
        # each element is an object of scores for each metric
        return_arr = []
        for rank in range(101):
            region_dict = {
                'rank': rank,
                'coordinates': top_region_coords[dl, rank],
                'region_ind': int(top_regions[dl, rank]),
                'score': float(roi_sim[dl, rank]),
                'percentiles': list(roi_per[dl, rank]),
                'factors': list(return_factor_order),
                'metrics': list(return_metric_order),
                'factor_colors': list(return_colors),
                'sentence': sentences[dl, rank],
                'detail_level': dl + 1
            }
            if rank == 0:
                region_dict['query_factor_order'] = query_factor_order.tolist()
                region_dict['target_factors'] = target_factors.tolist()
                region_dict['top_region_factor_Z_score'] = list(Z[dl])
                region_dict['top_region_factor_pval'] = list(pval[dl])
                # region_dict['roiURL'] = roiurl
            return_arr.append(region_dict)

        return_arr_full.append(return_arr)

    # print(time.time()-t)

    json_return = json.dumps(return_arr_full)
    # sys.stdout.write(json_return)
    json_return_all.append(json_return)


print(time.time()-t)

0
100
200
300
400
500
600
700
800
900
1263.1607818603516


In [16]:
# size in mb
sys.getsizeof(json_return_all) * 88 * 1e-6

0.793408